In [1]:
from transformers import AutoTokenizer  # Or BertTokenizer
from transformers import AutoModelForPreTraining  # Or BertForPreTraining for loading pretraining heads
from transformers import AutoModel  # or BertModel, for BERT without pretraining heads


e:\repos\pessoal\redem-index\studies\fake_news\.venv_new\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = AutoModelForPreTraining.from_pretrained('neuralmind/bert-base-portuguese-cased')
tokenizer = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased', do_lower_case=False)

e:\repos\pessoal\redem-index\studies\fake_news\.venv_new\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\caca_\.cache\huggingface\hub\models--neuralmind--bert-base-portuguese-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
from transformers import pipeline

pipe = pipeline('fill-mask', model=model, tokenizer=tokenizer)

Device set to use cpu


# Stance Classification with BERT

For stance classification, we need to use `AutoModelForSequenceClassification` instead of `AutoModelForPreTraining`. This model has a classification head on top of BERT.


In [7]:
from transformers import AutoModelForSequenceClassification
import torch

# Load model for sequence classification
# num_labels: number of stance classes (e.g., 3 for: support, oppose, neutral)
num_labels = 3  # Adjust based on your stance categories
classification_model = AutoModelForSequenceClassification.from_pretrained(
    'neuralmind/bert-base-portuguese-cased',
    num_labels=num_labels
)

# Set to evaluation mode
classification_model.eval()

print(f"Model loaded with {num_labels} labels for classification")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded with 3 labels for classification


## Option 1: Using the model directly (if already fine-tuned)

If you have a fine-tuned model, you can use it directly with the pipeline:


In [ ]:
# If you have a fine-tuned model saved, load it like this:
# fine_tuned_model = AutoModelForSequenceClassification.from_pretrained('./path/to/fine_tuned_model')
# stance_pipe = pipeline('text-classification', model=fine_tuned_model, tokenizer=tokenizer)

# Example usage (after fine-tuning):
# result = stance_pipe("Esta notícia é verdadeira e importante.")
# print(result)


## Option 2: Manual inference (for prediction with raw model)

This shows how to use the model for prediction manually:


In [8]:
def classify_stance(text, model, tokenizer, label_names=None):
    """
    Classify stance of a text.
    
    Args:
        text: Input text to classify
        model: Fine-tuned AutoModelForSequenceClassification
        tokenizer: Tokenizer
        label_names: Optional list of label names (e.g., ['support', 'oppose', 'neutral'])
    
    Returns:
        dict with predicted label and confidence scores
    """
    # Tokenize input
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    
    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    
    # Apply softmax to get probabilities
    probs = torch.nn.functional.softmax(logits, dim=-1)[0]
    
    # Get predicted class
    predicted_id = torch.argmax(probs).item()
    confidence = probs[predicted_id].item()
    
    result = {
        'text': text,
        'predicted_label': predicted_id,
        'confidence': confidence,
        'probabilities': probs.tolist()
    }
    
    # Add label names if provided
    if label_names:
        result['predicted_label_name'] = label_names[predicted_id]
        result['all_labels'] = {
            label_names[i]: float(probs[i]) 
            for i in range(len(label_names))
        }
    
    return result

# Example usage (requires fine-tuned model):
label_names = ['support', 'oppose', 'neutral']  # Adjust to your labels
result = classify_stance("Esta notícia é falsa.", classification_model, tokenizer, label_names)
print(result)


{'text': 'Esta notícia é falsa.', 'predicted_label': 0, 'confidence': 0.35822585225105286, 'probabilities': [0.35822585225105286, 0.2880688011646271, 0.35370534658432007], 'predicted_label_name': 'support', 'all_labels': {'support': 0.35822585225105286, 'oppose': 0.2880688011646271, 'neutral': 0.35370534658432007}}


## Option 3: Using as Feature Extractor + Custom Classifier

You can also use BERT as a feature extractor and add your own classifier:


In [9]:
from transformers import AutoModel
import torch.nn as nn

# Load base BERT model (without classification head)
base_model = AutoModel.from_pretrained('neuralmind/bert-base-portuguese-cased')

# Create a custom classifier
class StanceClassifier(nn.Module):
    def __init__(self, bert_model, num_labels=3, dropout=0.1):
        super().__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask=None):
        # Get BERT outputs
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] token representation
        pooled_output = outputs.pooler_output
        # Apply dropout and classification
        output = self.dropout(pooled_output)
        logits = self.classifier(output)
        return logits

# Create stance classifier
stance_classifier = StanceClassifier(base_model, num_labels=3)
stance_classifier.eval()

print("Custom stance classifier created")


Custom stance classifier created


## Option 4: Fine-tuning for Stance Classification

To fine-tune the model for your specific stance classification task:


In [ ]:
from transformers import TrainingArguments, Trainer
from torch.utils.data import Dataset
import pandas as pd

# Example dataset structure
class StanceDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Example training setup (commented out - uncomment and adjust for your data)
# """
# # Prepare your data
# # df should have columns: 'text' and 'label' (0, 1, 2 for support, oppose, neutral)
# train_texts = df_train['text'].tolist()
# train_labels = df_train['label'].tolist()
# 
# # Create datasets
# train_dataset = StanceDataset(train_texts, train_labels, tokenizer)
# 
# # Training arguments
# training_args = TrainingArguments(
#     output_dir='./stance_model',
#     num_train_epochs=3,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     warmup_steps=500,
#     weight_decay=0.01,
#     logging_dir='./logs',
#     logging_steps=10,
#     save_strategy='epoch',
#     evaluation_strategy='epoch' if val_dataset else 'no'
# )
# 
# # Initialize trainer
# trainer = Trainer(
#     model=classification_model,
#     args=training_args,
#     train_dataset=train_dataset,
#     # eval_dataset=val_dataset,  # if you have validation data
#     tokenizer=tokenizer,
# )
# 
# # Train
# trainer.train()
# 
# # Save model
# trainer.save_model('./stance_model_final')
# """

print("Training code template provided. Adjust according to your data.")


## Option 5: Extract Features from BERT (for traditional ML)

Even without fine-tuning, you can extract features from BERT for downstream classification:


In [11]:
def extract_bert_features(texts, model, tokenizer):
    """
    Extract BERT features (embeddings) from texts.
    These can be used with traditional ML classifiers (SVM, Random Forest, etc.)
    """
    import numpy as np
    features = []
    
    for text in texts:
        # Tokenize
        inputs = tokenizer(
            text, 
            return_tensors='pt', 
            truncation=True, 
            padding=True, 
            max_length=512
        )
        
        # Get BERT outputs
        with torch.no_grad():
            outputs = model(**inputs)
            # Use [CLS] token (first token) representation
            # outputs.last_hidden_state[:, 0, :] or use pooler_output if available
            if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
                feature = outputs.pooler_output[0].numpy()
            else:
                # Fallback: use [CLS] token
                feature = outputs.last_hidden_state[0, 0, :].numpy()
        
        features.append(feature)
    
    return np.array(features)

# Example usage:
sample_texts = [
    "Esta notícia é verdadeira.",
    "Esta informação é falsa."
]
features = extract_bert_features(sample_texts, base_model, tokenizer)
print(f"Extracted features shape: {features[0].shape}")
# Now you can use these features with sklearn classifiers:
from sklearn.svm import SVC
classifier = SVC()
classifier.fit(features, ['support', 'neutral'])


Extracted features shape: (768,)


,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


---

# Solution: Two-Step Approach for Post-Fake News Matching

## Step 1: Semantic Similarity (Find Related Posts)
Use embeddings to find which posts discuss topics covered in fact-check reports.

## Step 2: Stance Classification (Determine Post's Position)
Use BERT to classify whether the post supports, opposes, or is neutral regarding the fake news.


In [12]:
import chromadb
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple
from embeders import SentenceTransformerEmbeddingFunction

# Initialize ChromaDB client
client = chromadb.PersistentClient(path=r"./.chroma_db")

# Create embedding function (you can use Ollama or SentenceTransformer)
ef = SentenceTransformerEmbeddingFunction()

# Collections:
# - "checks" or "checks_v2": Contains fact-check reports (fake news debunks)
# - "posts" or "posts_cosine_sentence_transformer": Contains social media posts

try:
    factcheck_collection = client.get_collection(name="checks_v2_rewrite_cosine_sentence_transformer", embedding_function=ef)
    print("✓ Loaded fact-check reports collection")
except:
    try:
        factcheck_collection = client.get_collection(name="checks", embedding_function=ef)
        print("✓ Loaded fact-check reports collection (checks)")
    except:
        print("⚠ Fact-check collection not found. Please check collection name.")
        
# try:
#     posts_collection = client.get_collection(name="posts_cosine_sentence_transformer", embedding_function=ef)
#     print("✓ Loaded posts collection")
# except:
#     try:
#         posts_collection = client.get_collection(name="posts", embedding_function=ef)
#         print("✓ Loaded posts collection (posts)")
#     except:
#         print("⚠ Posts collection not found. Please check collection name.")


✓ Loaded fact-check reports collection


In [16]:
def find_posts_related_to_factcheck(
    post_text: str,
    factcheck_collection,
    similarity_threshold: float = 0.7,
    n_results: int = 5
) -> List[Dict]:
    """
    Find fact-check reports that are semantically similar to a social media post.
    
    Args:
        post_text: The social media post text
        factcheck_collection: ChromaDB collection containing fact-check reports
        similarity_threshold: Minimum cosine similarity score (0-1)
        n_results: Number of top similar reports to return
    
    Returns:
        List of dictionaries with fact-check report info and similarity scores
    """
    # Query the fact-check collection
    results = factcheck_collection.query(
        query_texts=[post_text],
        n_results=n_results
    )
    
    related_reports = []
    for i in range(len(results['ids'][0])):
        distance = results['distances'][0][i]
        # Convert distance to similarity (for cosine distance, similarity = 1 - distance)
        similarity = 1 - distance
        
        if similarity >= similarity_threshold:
            related_reports.append({
                'factcheck_id': results['ids'][0][i],
                'factcheck_content': results['documents'][0][i],
                'similarity_score': similarity,
                'distance': distance
            })
    
    return related_reports


# Example usage:
post = "A notícia diz que o presidente assinou um decreto proibindo vacinas"
related = find_posts_related_to_factcheck(post, factcheck_collection, similarity_threshold=0.6)
print(f"Found {len(related)} related fact-check reports")
for report in related:
    print(f"Similarity: {report['similarity_score']:.3f}")
    print(f"Report: {report['factcheck_content'][:200]}...\n")


Found 1 related fact-check reports
Similarity: 0.677
Report: Jair Bolsonaro foi condenado nos EUA e investigado pelo FBI por fraudar o cartão de vacinação contra Covid-19...



## Step 2: Classify Stance Using BERT

Now that we've found related fact-check reports, we need to determine if the post:
- **Supports** the fake news (shares/promotes it)
- **Opposes** the fake news (debunks/criticizes it)
- **Is neutral** (just mentions it without clear stance)


In [17]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load Portuguese BERT for stance classification
stance_model_name = 'neuralmind/bert-base-portuguese-cased'
num_stance_labels = 3  # support, oppose, neutral

# Load tokenizer
stance_tokenizer = AutoTokenizer.from_pretrained(stance_model_name, do_lower_case=False)

# For this example, we'll use the classification model from earlier
# In production, you'd load a fine-tuned model:
# stance_model = AutoModelForSequenceClassification.from_pretrained('./path/to/fine_tuned_stance_model')
# For now, using untrained model (random weights - needs fine-tuning!)
stance_model = AutoModelForSequenceClassification.from_pretrained(
    stance_model_name,
    num_labels=num_stance_labels
)
stance_model.eval()

print("✓ Stance classification model loaded")
print("⚠ Note: Model needs fine-tuning on your data for accurate results!")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Stance classification model loaded
⚠ Note: Model needs fine-tuning on your data for accurate results!


In [37]:
def classify_post_stance(
    post_text: str,
    factcheck_report: str,
    model,
    tokenizer,
    label_names: List[str] = ['support', 'oppose', 'neutral']
) -> Dict:
    """
    Classify the stance of a social media post regarding a fact-check report.
    
    Args:
        post_text: The social media post
        factcheck_report: The related fact-check report content
        model: Fine-tuned BERT model for stance classification
        tokenizer: BERT tokenizer
        label_names: List of stance label names
    
    Returns:
        Dictionary with stance prediction and confidence scores
    """
    # Create input: combine post and fact-check context
    # Format: "[CLS] post [SEP] factcheck_report [SEP]"
    combined_text = f"{post_text} [SEP] {factcheck_report}"
    
    # Tokenize
    inputs = tokenizer(
        combined_text,
        post_text,  # Only encode post for now (or use combined)
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=512
    )
    
    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    
    # Apply softmax to get probabilities
    probs = torch.nn.functional.softmax(logits, dim=-1)[0]
    
    # Get predicted class
    predicted_id = torch.argmax(probs).item()
    confidence = probs[predicted_id].item()
    
    result = {
        'predicted_stance': label_names[predicted_id],
        'confidence': float(confidence),
        'all_scores': {
            label_names[i]: float(probs[i]) 
            for i in range(len(label_names))
        },
        'is_sharing_fake_news': label_names[predicted_id] == 'support'
    }
    
    return result

# Example usage:
post = "Acabei de ler essa notícia sobre vacinas, compartilhem!"
factcheck = "Verificação: A notícia sobre proibição de vacinas é falsa. O decreto não existe."
stance = classify_post_stance(post, factcheck, stance_model, stance_tokenizer)
print(stance)


{'predicted_stance': 'oppose', 'confidence': 0.4241788387298584, 'all_scores': {'support': 0.3431606888771057, 'oppose': 0.4241788387298584, 'neutral': 0.2326604574918747}, 'is_sharing_fake_news': False}


## Complete Pipeline: End-to-End Analysis

This function combines both steps to analyze a post completely.


In [19]:
def analyze_post_against_fake_news(
    post_text: str,
    factcheck_collection,
    stance_model,
    stance_tokenizer,
    similarity_threshold: float = 0.6,
    n_reports: int = 3,
    label_names: List[str] = ['support', 'oppose', 'neutral']
) -> Dict:
    """
    Complete pipeline: Find related fact-checks and classify stance.
    
    Args:
        post_text: Social media post to analyze
        factcheck_collection: ChromaDB collection with fact-check reports
        stance_model: Fine-tuned BERT model for stance classification
        stance_tokenizer: BERT tokenizer
        similarity_threshold: Minimum similarity to consider a match
        n_reports: Number of fact-check reports to check
        label_names: Stance label names
    
    Returns:
        Dictionary with analysis results
    """
    # Step 1: Find related fact-check reports
    related_reports = find_posts_related_to_factcheck(
        post_text,
        factcheck_collection,
        similarity_threshold=similarity_threshold,
        n_results=n_reports
    )
    
    if not related_reports:
        return {
            'post': post_text,
            'related_factchecks': [],
            'most_likely_stance': None,
            'is_related_to_fake_news': False,
            'message': 'No related fact-check reports found'
        }
    
    # Step 2: Classify stance for each related report
    stance_analyses = []
    for report in related_reports:
        stance_result = classify_post_stance(
            post_text,
            report['factcheck_content'],
            stance_model,
            stance_tokenizer,
            label_names
        )
        
        stance_analyses.append({
            'factcheck_id': report['factcheck_id'],
            'factcheck_content': report['factcheck_content'][:200] + '...',
            'similarity_score': report['similarity_score'],
            'stance': stance_result['predicted_stance'],
            'confidence': stance_result['confidence'],
            'is_sharing_fake_news': stance_result['is_sharing_fake_news']
        })
    
    # Get the most confident stance prediction
    best_match = max(stance_analyses, key=lambda x: x['confidence'])
    
    return {
        'post': post_text,
        'related_factchecks_count': len(related_reports),
        'stance_analyses': stance_analyses,
        'most_likely_stance': best_match['stance'],
        'most_likely_confidence': best_match['confidence'],
        'is_related_to_fake_news': True,
        'is_sharing_fake_news': best_match['is_sharing_fake_news']
    }


# Example usage:
result = analyze_post_against_fake_news(
    post_text="Vejam essa notícia importante sobre políticas de saúde",
    factcheck_collection=factcheck_collection,
    stance_model=stance_model,
    stance_tokenizer=stance_tokenizer
)
print(result)


{'post': 'Vejam essa notícia importante sobre políticas de saúde', 'related_factchecks': [], 'most_likely_stance': None, 'is_related_to_fake_news': False, 'message': 'No related fact-check reports found'}


In [20]:
from tqdm import tqdm

def batch_analyze_posts(
    posts: List[str],
    factcheck_collection,
    stance_model,
    stance_tokenizer,
    similarity_threshold: float = 0.6,
    n_reports: int = 3
) -> pd.DataFrame:
    """
    Analyze multiple posts in batch.
    
    Args:
        posts: List of post texts to analyze
        factcheck_collection: ChromaDB collection with fact-check reports
        stance_model: Fine-tuned BERT model
        stance_tokenizer: BERT tokenizer
        similarity_threshold: Minimum similarity threshold
        n_reports: Number of fact-check reports to consider
    
    Returns:
        DataFrame with analysis results
    """
    results = []
    
    for post in tqdm(posts, desc="Analyzing posts"):
        try:
            analysis = analyze_post_against_fake_news(
                post,
                factcheck_collection,
                stance_model,
                stance_tokenizer,
                similarity_threshold=similarity_threshold,
                n_reports=n_reports
            )
            results.append(analysis)
        except Exception as e:
            print(f"Error analyzing post '{post[:50]}...': {e}")
            results.append({
                'post': post,
                'error': str(e)
            })
    
    return pd.DataFrame(results)


# Example usage:
sample_posts = [
    "Notícia importante sobre saúde pública!",
    "Mais uma fake news sendo desmentida pelos fatos",
    "Vi essa informação e preciso compartilhar"
]

df_results = batch_analyze_posts(
    sample_posts,
    factcheck_collection,
    stance_model,
    stance_tokenizer
)
print(df_results)


Analyzing posts: 100%|██████████| 3/3 [00:00<00:00,  3.55it/s]

                                              post related_factchecks  \
0          Notícia importante sobre saúde pública!                 []   
1  Mais uma fake news sendo desmentida pelos fatos                 []   
2        Vi essa informação e preciso compartilhar                 []   

  most_likely_stance  is_related_to_fake_news  \
0               None                    False   
1               None                    False   
2               None                    False   

                               message  
0  No related fact-check reports found  
1  No related fact-check reports found  
2  No related fact-check reports found  


# Compare to actual posts

In [21]:
df = pd.read_csv("df_res.csv")
print(f"Total de mensagens: {len(df)}")

Total de mensagens: 91648


In [38]:
sample_posts = df.sort_values("min_distance")["message"].to_list()[:10]
df_results = batch_analyze_posts(
    sample_posts,
    factcheck_collection,
    stance_model,
    stance_tokenizer
)

Analyzing posts: 100%|██████████| 10/10 [00:08<00:00,  1.24it/s]


In [ ]:
for i, row in df_results.iterrows():
    print(row["post"])
    print('-' * 100)
    print(row["most_likely_stance"])
    print('-' * 100)
    print(row["stance_analyses"])
    print("=" * 100)


A que ponto chegamos quando é necessário destacar na Lei de Diretrizes Orçamentárias que a União não pode realizar despesas com ações que incentivem a invasão de propriedade privada, promovam opções sexuais diferentes do sexo biológico para crianças e adolescentes, ou que diminuam ou desconstruam o conceito de família tradicional. Sem contar o absurdo da cirurgia de mudança de sexo em crianças e adolescentes. Vamos continuar lutando sempre pelo que é certo.

#plnacional22 #riograndedosul #patriotas #pl22rs #camaradosdeputados  #direitapatriota
----------------------------------------------------------------------------------------------------
oppose
----------------------------------------------------------------------------------------------------
[{'factcheck_id': '19', 'factcheck_content': 'mudança de sexo em crianças com verba da União...', 'similarity_score': 0.6458995342254639, 'stance': 'oppose', 'confidence': 0.4456923305988312, 'is_sharing_fake_news': False}]
??Nas fotos compa

## Next Steps: Fine-tuning the Stance Classifier

To improve accuracy, you need to fine-tune the stance classifier on your data:

1. **Collect training data**: Annotate posts with stance labels (support/oppose/neutral)
2. **Create dataset**: Use the `StanceDataset` class from Option 4 above
3. **Train model**: Fine-tune BERT on your annotated data
4. **Evaluate**: Test on held-out data

Example data format:
```python
training_data = [
    {"text": "Compartilhem essa notícia urgente!", "label": 0},  # support
    {"text": "Essa informação foi desmentida pelos fatos", "label": 1},  # oppose
    {"text": "Vi uma discussão sobre esse tema", "label": 2},  # neutral
]
```
